# 06 - Quasi-probability 워크플로

목표: Z-모멘트로부터 근사 확률을 계산합니다.

quasi-probability란?
- 비트열(bitstring)에 대해 정의된, 부호를 가질 수 있는 유사 확률 분포 함수입니다.
- 음수 값을 가질 수 있으므로 그 자체로는 직접 샘플링할 수 있는 물리적 확률 분포가 아닙니다.
- 여기서는 차수를 절단한 모멘트 전개로 구성합니다:
  `q^(k)(x) = 2^{-n} * sum_{S: |S|<=k} (-1)^{sum_{i in S} x_i} <Z_S>`

직관적 이해:
- `Z`-모멘트(`<Z_S>`)는 선택된 큐비트 부분집합의 상관관계를 나타냅니다.
- `k`가 작으면 저차 상관항만 유지합니다(계산은 빠르지만 근사치).
- `k`를 키우면 고차 상관항까지 포함됩니다(더 정확하지만 계산 비용 증가).
- 최대 차수 `n`까지의 모든 부분집합을 포함하면 `q^(n)`은 정확한 확률 분포를 복원합니다(검증용 소규모 시스템 기준).

주요 객체:
- `z_combos`: 사용할 상관자의 큐비트 부분집합 목록
- `moments`: `z_combos` 순서에 맞춰 계산된 `<Z_S>` 기댓값
- `q^(k)`: 차수 `k`까지 사용해 복원한 quasi-probability

In [1]:
import warnings
warnings.filterwarnings("ignore")

import torch

from padopauli import Circuit, build_quasi_sampler


In [2]:
from itertools import combinations

n_qubits = 4
max_order = 4

qc = Circuit(n_qubits=n_qubits)
qc.h(0)
qc.cnot(0, 1)
qc.rzz(1, 2, param_idx=0)
qc.rxx(2, 3, param_idx=1)

z_combos = []
for k in range(1, max_order + 1):
    z_combos.extend([list(c) for c in combinations(range(n_qubits), k)])

# build_quasi_sampler는 게이트 리스트(qc.gates)를 입력으로 받습니다.
sampler = build_quasi_sampler(
    n_qubits=n_qubits,
    circuit=qc.gates,
    z_combos=z_combos,
    max_order=max_order,
    preset="gpu",
)

propagate:   0%|          | 0/4 [00:00<?, ?it/s]

[PPS Info] Propagation complete. Terms generated: 7


zero-filter:   0%|          | 0/3 [00:00<?, ?it/s]

[PPS Info] Terms retained after pruning: 7 (100.000000% of peak)


## `q^(k)(x)` 값의 해석 방법

- `q^(k)(x) > 0`: 유지된 상관항들이 상태 `x`에 양의 확률로 기여함을 뜻합니다.
- `q^(k)(x) < 0`: 절단으로 인해 생기는 오차(생략된 고차 항들의 간섭 효과)입니다.
- 모멘트들이 일관되게 계산되었다면 전체 합 `sum_x q^(k)(x) = 1`은 항상 성립합니다.

직접 샘플링할 수 없고 음수가 나올 수 있음에도 유용한 이유:
- 학습에 활용할 수 있는 미분 가능한 surrogate 목적함수를 제공합니다.
- 선택한 차수까지의 저차 통계량을 정확히 보존합니다.
- `k`를 늘릴수록 일반적으로 음수 영역이 줄어들며 `q^(k)`가 실제 확률 분포에 수렴합니다.

필수 확인 사항:
1. 확률 총합 `sum(q)`가 1에 가까운지 확인
2. 정확한 기준값 대비 개별 모멘트의 오차 확인
3. 소규모 큐비트(n) 예제에서 전체 분포 오차(`max|q-p|`, `L1`) 확인

이 노트북에서는 모멘트 계산 오차, 복원 과정의 오차, 그리고 비트 인덱싱·순서 오류를 명확히 구분해 확인할 수 있도록 세 가지 지표를 모두 출력합니다.

## 모멘트 계산 및 정확한 확률 분포와의 비교 검증

이 섹션에서는 다음 세 가지를 단계별로 검증합니다.
1. 모멘트 자체의 계산 정확도
2. 모멘트로부터 `q(x)`를 복원하는 공식의 내부 일관성
3. PennyLane으로 구한 정확한 확률 분포와의 최종 일치 여부

In [3]:
thetas = torch.tensor([0.3, -0.4], dtype=torch.float64)

moments = sampler.compute_moments(thetas)
print("moments shape:", tuple(moments.shape))

moments shape: (1, 15)


In [4]:
import numpy as np
import pennylane as qml

order_eval = 4

def _apply_circuit_pennylane(circuit, thetas_np):
    for gate in circuit:
        name = type(gate).__name__
        if name == "CliffordGate":
            symbol = gate.symbol
            if symbol == "H":
                qml.Hadamard(wires=gate.qubits[0])
            elif symbol == "S":
                qml.S(wires=gate.qubits[0])
            elif symbol == "X":
                qml.PauliX(wires=gate.qubits[0])
            elif symbol == "Y":
                qml.PauliY(wires=gate.qubits[0])
            elif symbol == "Z":
                qml.PauliZ(wires=gate.qubits[0])
            elif symbol == "CNOT":
                qml.CNOT(wires=gate.qubits)
            elif symbol == "CZ":
                qml.CZ(wires=gate.qubits)
            elif symbol == "SWAP":
                qml.SWAP(wires=gate.qubits)
            else:
                raise ValueError(f"Unsupported CliffordGate symbol for PennyLane: {symbol}")
        elif name == "PauliRotation":
            theta = float(thetas_np[int(gate.param_idx)])
            qml.PauliRot(theta, gate.pauli, wires=gate.qubits)
        else:
            raise ValueError(f"Unsupported gate type for PennyLane: {name}")

dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def exact_probs(thetas_np):
    _apply_circuit_pennylane(qc.gates, thetas_np)
    return qml.probs(wires=list(range(n_qubits)))

def _bits_le_from_int(x, n):
    return [((x >> i) & 1) for i in range(n)]

def _bit_reverse(x, n):
    y = 0
    for i in range(n):
        y = (y << 1) | ((x >> i) & 1)
    return y

thetas_np = thetas.detach().cpu().numpy().astype(np.float64)
p_be = np.asarray(exact_probs(thetas_np), dtype=np.float64)

# 샘플러는 little-endian 큐비트 순서를 따릅니다: 즉, row[i]가 큐비트 i의 비트값에 해당합니다.
all_rows = np.array([_bits_le_from_int(code, n_qubits) for code in range(2 ** n_qubits)], dtype=np.uint8)
all_bits = torch.tensor(all_rows, dtype=torch.uint8, device=moments.device)

q_all = sampler.quasi_prob_from_moments(all_bits, moments, order=order_eval).detach().cpu().numpy().astype(np.float64)
p_all = np.array([p_be[_bit_reverse(code, n_qubits)] for code in range(2 ** n_qubits)], dtype=np.float64)

diff = q_all - p_all
print(f"order={order_eval}, max|q-p|={np.max(np.abs(diff)):.6e}, L1={np.sum(np.abs(diff)):.6e}")
print(f"sum(q)={np.sum(q_all):.12f}, sum(p)={np.sum(p_all):.12f}")


# 내부 일관성 확인: 벡터화된 q 계산 결과와 명시적 루프 계산 결과 비교
def q_from_moments_ref(bits_le, moments_np, combos, order, n):
    out = []
    for row in bits_le:
        acc = 1.0
        for m, sub in zip(moments_np, combos):
            if len(sub) > order:
                continue
            sign = 1.0
            for q in sub:
                sign *= (1.0 if int(row[q]) == 0 else -1.0)
            acc += sign * float(m)
        out.append(acc / float(2 ** n))
    return np.asarray(out, dtype=np.float64)

moments_np = moments.detach().cpu().numpy().astype(np.float64).reshape(-1)
q_ref = q_from_moments_ref(all_rows, moments_np, z_combos, order_eval, n_qubits)
print(f"internal q check: max|q_vec-q_ref|={np.max(np.abs(q_all - q_ref)):.6e}")


# 오차 원인을 구체적으로 파악하기 위한 개별 모멘트 비교
moments_np = moments.detach().cpu().numpy().astype(np.float64).reshape(-1)
m_exact = []
for sub in z_combos:
    acc = 0.0
    for code in range(2 ** n_qubits):
        sign = 1.0
        for q in sub:
            sign *= (1.0 if ((code >> q) & 1) == 0 else -1.0)
        acc += sign * p_all[code]
    m_exact.append(acc)
m_exact = np.asarray(m_exact, dtype=np.float64)
mdiff = moments_np - m_exact
print(f"moments: max|dm|={np.max(np.abs(mdiff)):.6e}, mean|dm|={np.mean(np.abs(mdiff)):.6e}")

print("\nTop states by p_exact:")
top = np.argsort(-p_all)
for i in top[:8]:
    b = "".join(str(int(v)) for v in all_rows[i])
    print(f"  x={b}  p_exact={p_all[i]: .6e}  q={q_all[i]: .6e}  (q-p)={diff[i]: .3e}")

order=4, max|q-p|=6.106227e-16, L1=1.262879e-15
sum(q)=1.000000000000, sum(p)=1.000000000000
internal q check: max|q_vec-q_ref|=0.000000e+00
moments: max|dm|=1.221245e-15, mean|dm|=5.699145e-16

Top states by p_exact:
  x=0000  p_exact= 4.802652e-01  q= 4.802652e-01  (q-p)= 6.106e-16
  x=1100  p_exact= 4.802652e-01  q= 4.802652e-01  (q-p)= 6.106e-16
  x=0011  p_exact= 1.973475e-02  q= 1.973475e-02  (q-p)= 2.082e-17
  x=1111  p_exact= 1.973475e-02  q= 1.973475e-02  (q-p)= 2.082e-17
  x=0010  p_exact= 0.000000e+00  q= 0.000000e+00  (q-p)= 0.000e+00
  x=1010  p_exact= 0.000000e+00  q= 0.000000e+00  (q-p)= 0.000e+00
  x=1000  p_exact= 0.000000e+00  q= 0.000000e+00  (q-p)= 0.000e+00
  x=0100  p_exact= 0.000000e+00  q= 0.000000e+00  (q-p)= 0.000e+00


### 단일 호출 API
`sampler.quasi_prob(x_batch, thetas, order=...)`를 호출하면
1) 모멘트 계산과 2) quasi-probability 복원을 한 번에 수행합니다.

In [5]:
for k in (1, 2, 4):
    q_k = sampler.quasi_prob(all_bits, thetas, order=k).detach().cpu().numpy()
    print(f"order={k}: L1(q-p)={np.sum(np.abs(q_k - p_all)):.3e}, min(q)={q_k.min(): .3e}")


order=1: L1(q-p)=1.711e+00, min(q)=-5.263e-02
order=2: L1(q-p)=1.421e+00, min(q)=-6.250e-02
order=4: L1(q-p)=1.263e-15, min(q)= 0.000e+00
